# Исследование: Моделирование изменения уровня жидкости в резервуарах с учетом давления и потока

## Введение

В данном исследовании рассматривается метод моделирования изменения уровня жидкости в резервуарах, которые соединены трубопроводной системой. Мы будем учитывать давление в системе и использовать его для расчета потока жидкости между резервуарами. Поток жидкости вычисляется через разницу напоров между резервуарами, используя закон Бернулли. Все изменения происходят мгновенно, и уровни жидкости обновляются на основе текущих значений давления.

## Математическая модель

Для каждого резервуара будем использовать следующие уравнения:

### Эффективный напор

Эффективный напор $ H $ для каждого резервуара вычисляется как сумма уровня жидкости и давления, преобразованного в высоту с использованием уравнения:

$$
H_i = h_i + \frac{p_i}{\rho g}
$$

где:
- $ h_i $ — уровень жидкости в резервуаре $ i $ (в метрах),
- $ p_i $ — давление в резервуаре $ i $ (в паскалях),
- $ \rho $ — плотность жидкости (в кг/м³),
- $ g $ — ускорение свободного падения (в м/с²).

### Поток жидкости

Поток жидкости $ Q $ между двумя резервуарами $ i $ и $ j $ вычисляется с использованием формулы Бернулли для несжимаемой жидкости:

$$
Q_{ij} = A_{ij} \sqrt{2 g \left| H_i - H_j \right|}
$$

где:
- $ A_{ij} $ — площадь сечения трубопровода между резервуарами $ i $ и $ j $ (в м²),
- $ H_i $ и $ H_j $ — эффективные напоры для резервуаров $ i $ и $ j $,
- $ g $ — ускорение свободного падения.

### Обновление уровня жидкости

После вычисления потока жидкости, мы обновляем уровни жидкости в резервуарах с учетом изменения объема:

$$
h_{\text{new}} = h_{\text{old}} + \frac{F}{A_{\text{res}}}
$$

где:
- $ F $ — суммарный поток в резервуаре,
- $ A_{\text{res}} $ — площадь сечения резервуара.

Новые уровни жидкости ограничены максимальными уровнями, которые вычисляются как $ \frac{V}{A_{\text{res}}} $, где $ V $ — объем резервуара.

## Реализация модели

Теперь представим Python код, который моделирует изменение уровней жидкости в резервуарах с учетом давления и потока:




In [ ]:

import numpy as np
import math

def calculate_liquid_level(volumes, pressures, liquid_level_init, matrix):
    # Параметры
    rho = 1000.0  # плотность воды, кг/м³
    g = 9.81  # ускорение свободного падения, м/с²

    # Расчет эффективного напора
    N = len(liquid_level_init)  # количество резервуаров
    h = np.array(liquid_level_init, dtype=float)  # начальные уровни жидкости
    p = np.array(pressures, dtype=float)  # давления
    A_res = (
        np.array(volumes, dtype=float) / h
    )  # площади сечения резервуаров (A = V / h)

    # Матрица соединений (A_conn)
    A_conn = np.array(matrix, dtype=float) * 0.01  # преобразуем площадь из дм² в м²

    # Эффективный напор
    H = h + p / (rho * g)  # эффективный напор H = h + p / (rho * g)

    # Потоки между резервуарами
    F = np.zeros(N, dtype=float)  # Массив для хранения потоков

    for i in range(N):
        for j in range(N):  # Проверяем все соединения
            if i != j and A_conn[i, j] > 0:
                head_diff = H[i] - H[j]  # Разница напоров
                if abs(head_diff) < 1e-6:  # Если разница напоров мала, поток нулевой
                    Q = 0.0
                else:
                    Q = A_conn[i, j] * math.sqrt(
                        2 * g * abs(head_diff)
                    )  # Расчет потока

                # Направление потока
                if head_diff > 0:
                    F[i] -= Q
                    F[j] += Q
                elif head_diff < 0:
                    F[i] += Q
                    F[j] -= Q

    # Изменение уровней жидкости
    dh_dt = F / A_res  # Изменение уровня жидкости (dh/dt = net_flow / A_res)
    h_new = h + dh_dt  # Новый уровень жидкости

    # Обрезаем уровень жидкости, чтобы он не выходил за пределы объёмов резервуаров
    h_new = np.clip(h_new, 0, volumes)  # Уровни не могут превышать объём резервуара

    return h_new

## Моделирование системы, на тестовых данных